# Azure Databricks를 사용하여 데이터 모델링 설계 및 구현

## 업계 상황: 금융 서비스
이 데모에서는 **FinServ Demo**라는 **가상의 금융 서비스 회사**를 사용합니다. 이 회사는 고객 포트폴리오를 관리하고 자산 클래스(주식, 채권, ETF 및 FX) 전반에 걸쳐 거래를 실행하며 실시간 시장 데이터를 유지 관리하는 투자 은행입니다.

데이터 모델은 다음과 같이 정렬됩니다.
- **클라이언트** — 미주, EMEA, APAC 전역에 걸쳐 각각 KYC 상태, 위험 등급 및 부문을 갖춘 400명의 클라이언트
- **계좌** — 고객별 당좌, 저축, 투자, 연금, 거래 계좌
- **거래** — 2년간 8,000건 이상의 주식, 채권, ETF 및 FX 거래
- **시장 데이터** — 1년 동안 26개 증권에 대한 일일 OHLCV 가격 데이터

이러한 맥락에서 데이터 모델링 결정은 규제 쿼리를 위한 거래 날짜별 분할, 고객 위험 등급 변경을 위한 SCD 유형 2, 감사 추적을 위한 변경 데이터 피드, 다차원 거래 분석을 위한 유동 클러스터링 등 데이터 모델링 결정을 실질적으로 만듭니다.

In [0]:
from scripts.setup import setup, setup_06

spark.sql("USE CATALOG trainer_demo")
setup_06(spark)

## 수집 로직 및 데이터 소스 구성 설계

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-data-ingestion-logic.png)

### 토론: 무역 데이터의 추출 패턴
FinServ Demo는 주문 관리 시스템(OMS)에서 거래 데이터를 수집합니다. 거래는 소스에 따라 다양한 방식으로 이루어집니다.
 패턴 | 사용 시점 | 금융 예시 |
---|---|---|
 **전체 추출** | 작은 참조 테이블, 변경 추적 없음 | 시장 참조 데이터 (통화 코드, 거래소 코드) |
 **증분 추출** | 신뢰할 수 있는 타임스탬프 컬럼이 있는 대형 테이블 | 거래 및 결제 — `trade_date`가 워터마크 |
 **스트리밍 추출** | 실시간 이벤트 처리, 틱 데이터 | 시장 가격 피드, 장중 주문장 업데이트 |

## 데이터 수집 도구 선택

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-data-ingestion-tool.png)

### 레이크플로우 커넥트
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-lakeflow-connect.png)

### 자동 로더
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-auto-loader.png)

### 복사 대상
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-copy-into.png)

### Azure 데이터 팩토리
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-azure-data-factory.png)

### 의사결정 프레임워크 적용
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-decision-framework.png)

## 데이터 테이블 형식을 선택하세요

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-data-table-format.png)

### Parquet가 데이터를 다르게 저장하는 방법
![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/4-parquet-file-layout.png)

## 데이터 분할 방식 설계 및 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-data-partitioning-scheme.png)

### 🎯 데모: 거래 날짜별로 거래 테이블 분할하기
규정 준수 팀은 정기적으로 FinServ 데모 거래를 날짜 범위별로 쿼리합니다. "2023-Q1과 2023-Q2 사이의 모든 거래를 표시합니다". 분할 없이 모든 쿼리는 8,000개의 거래 기록을 모두 검색합니다. `trade_date`에 의한 일일 파티셔닝을 통해 엔진은 관련 날짜 폴더만 읽습니다.
> **분할 시기**: 이 데모에서는 거래 테이블에 8,000개의 행이 있습니다. 이는 1TB Databricks 임계값보다 훨씬 적습니다. FinServ Demo의 프로덕션(수년간 수백만 건의 거래)에서는 `trade_date`에 의한 분할이 필수적입니다.
> **유체 클러스터링**은 최신 권장 사항입니다. 클러스터링 데모에서 이에 대해 살펴보겠습니다.

In [0]:
%sql
-- Create a partitioned version of the trades table.
-- PARTITIONED BY (trade_date) puts each day's trades in a separate directory.
-- The query engine uses partition pruning to skip directories that don't match the filter.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE trades_partitioned
USING DELTA
PARTITIONED BY (trade_date)
AS
SELECT * FROM trades;

-- Show partition statistics
DESCRIBE EXTENDED trades_partitioned;


In [0]:
%sql
-- Query with a date filter — the engine prunes all partition folders outside this range.
-- EXPLAIN shows the physical plan and confirms partition pruning is applied.

EXPLAIN
SELECT ticker, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_partitioned
WHERE trade_date BETWEEN '2023-01-01' AND '2023-03-31'
GROUP BY ticker
ORDER BY total_value DESC;


In [0]:
%sql
-- Generated columns: derive the partition column from a timestamp.
-- In many OMS exports, trades include a precise timestamp. Using a generated
-- column, you can partition by date while keeping the full timestamp.

CREATE OR REPLACE TABLE trades_with_timestamp
(
    trade_id        STRING NOT NULL,
    account_id      STRING,
    ticker          STRING,
    asset_class     STRING,
    trade_type      STRING,
    quantity        INT,
    price           DOUBLE,
    total_value     DOUBLE,
    trade_timestamp TIMESTAMP,
    trade_date      DATE GENERATED ALWAYS AS (CAST(trade_timestamp AS DATE)),
    settlement_date DATE,
    status          STRING,
    region          STRING
)
USING DELTA
PARTITIONED BY (trade_date);

-- trade_date is auto-populated from trade_timestamp — no redundant column to maintain
DESCRIBE trades_with_timestamp;


## SCD(천천히 변화하는 측정기준) 유형을 선택하세요.

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-scd-type.png)

### SCD 유형 비교
다음 표에는 각 SCD 유형을 선택하는 경우가 요약되어 있습니다.
 고려사항                   | 유형 0                        | 유형 1                             | 유형 2                                | 유형 3                   |
 --------------------------- | ----------------------------- | ---------------------------------- | ------------------------------------- | ------------------------ |
 **이력 보존**              | 원본만                        | 없음                               | 전체                                  | 제한적                  |
 **저장소 영향**            | 최소                          | 최소                               | 높음                                  | 중간                    |
 **쿼리 복잡성**            | 단순                          | 단순                               | 복잡                                  | 중간                    |
 **일반 사용 사례**         | 상수, 감사 요구사항           | 오류 수정, 연락처 업데이트          | 판매 귀속, 지역 추적                  | 전후 비교               |
 **팩트 테이블 관계**       | 자연 키                       | 자연 키                            | 대리 키                               | 자연 키                  |

## SCD(천천히 변경되는 차원) 유형 2 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.implement-slowly-changing-dimension-type-2.png)


### 🎯 데모: FinServ 데모 클라이언트용 SCD 유형 2 구현
FinServ Demo의 고객 기록은 시간이 지남에 따라 두 가지 중요한 방식으로 변경됩니다.
- **지역** — 고객이 거주지를 옮기면 EMEA에서 미주 지역으로 재배정됩니다.
- **위험 등급** — 신용 검토를 통해 고객의 위험 등급이 낮음에서 높음으로 변경됩니다.

규정 준수 및 포트폴리오 할당을 위해서는 **각 거래 시점**의 고객 위험 등급을 알아야 합니다. SCD Type 2는 기존 기록을 덮어쓰는 대신 새 버전 행을 삽입하여 이 기록을 보존합니다.

`client_staging` 테이블은 CRM 시스템(업스트림 소스)의 최신 데이터를 시뮬레이션합니다. 이 테이블에는 다음이 포함됩니다.
- **60개의 업데이트된 레코드**(기존 고객의 경우 지역 또는 위험 등급이 변경됨)
- **10명의 신규 클라이언트** (아직 차원 테이블에 없는 신규 고객)

In [0]:
%sql
-- Step 1: Create the client dimension table with SCD Type 2 tracking columns.
-- Surrogate key (client_sk) uniquely identifies each version.
-- Natural key (client_id) links versions to the same real-world client.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE client_dim (
    client_sk            BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
    client_id            STRING NOT NULL,                        -- natural/business key
    first_name           STRING,
    last_name            STRING,
    email                STRING,
    country              STRING,
    region               STRING,
    segment              STRING,
    kyc_status           STRING,
    risk_rating          STRING,
    relationship_manager STRING,
    onboarding_date      DATE,
    valid_from           TIMESTAMP NOT NULL,
    valid_to             TIMESTAMP NOT NULL,
    is_current           BOOLEAN
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Initial load: treat all clients as current versions
INSERT INTO client_dim (
    client_id,
    first_name,
    last_name,
    email,
    country,
    region,
    segment,
    kyc_status,
    risk_rating,
    relationship_manager,
    onboarding_date,
    valid_from,
    valid_to,
    is_current
)
SELECT
    client_id, first_name, last_name, email, country,
    region, segment, kyc_status, risk_rating, relationship_manager,
    onboarding_date,
    CAST('2020-01-01' AS TIMESTAMP)    AS valid_from,
    CAST('9999-12-31' AS TIMESTAMP)    AS valid_to,
    true                               AS is_current
FROM trainer_demo.demo_06.clients;

SELECT COUNT(*) AS initial_rows FROM client_dim;


In [0]:
%sql
-- Step 2: Apply changes from the staging table using a two-step MERGE pattern.
-- Step A — close the current row for records that changed (set valid_to and is_current = false)

MERGE INTO trainer_demo.demo_06.client_dim AS target
USING (
    SELECT
        s.client_id,
        s.region,
        s.risk_rating,
        s.kyc_status,
        s.relationship_manager,
        current_timestamp() AS effective_ts
    FROM trainer_demo.demo_06.client_staging AS s
) AS updates
ON target.client_id = updates.client_id
   AND target.is_current = true
WHEN MATCHED AND (
    target.region               <> updates.region OR
    target.risk_rating          <> updates.risk_rating OR
    target.kyc_status           <> updates.kyc_status OR
    target.relationship_manager <> updates.relationship_manager
) THEN UPDATE SET
    target.valid_to    = updates.effective_ts,
    target.is_current  = false;

SELECT COUNT(*) AS expired_rows FROM trainer_demo.demo_06.client_dim WHERE is_current = false;


In [0]:
%sql
-- Step B: Insert new current versions for records that were just closed,
-- and also insert brand-new clients that don't yet exist in the dimension.

INSERT INTO trainer_demo.demo_06.client_dim (
    client_id,
    first_name,
    last_name,
    email,
    country,
    region,
    segment,
    kyc_status,
    risk_rating,
    relationship_manager,
    onboarding_date,
    valid_from,
    valid_to,
    is_current
)
SELECT
    s.client_id, s.first_name, s.last_name, s.email,
    s.country, s.region, s.segment, s.kyc_status,
    s.risk_rating, s.relationship_manager, s.onboarding_date,
    current_timestamp()            AS valid_from,
    CAST('9999-12-31' AS TIMESTAMP) AS valid_to,
    true                            AS is_current
FROM trainer_demo.demo_06.client_staging AS s
LEFT JOIN trainer_demo.demo_06.client_dim AS d
    ON s.client_id = d.client_id AND d.is_current = true
WHERE d.client_id IS NULL   -- new client (not in dim yet)
   OR (                     -- existing client whose current row was just closed
       d.client_id IS NOT NULL
       AND d.is_current = false
   );

SELECT
    is_current,
    COUNT(*) AS rows
FROM trainer_demo.demo_06.client_dim
GROUP BY is_current
ORDER BY is_current;


In [0]:
%sql
-- Step C: Query the history of a specific client to see risk rating changes over time.
-- This is the SCD Type 2 "complete history" query pattern.

SELECT
    client_id,
    first_name || ' ' || last_name AS client_name,
    risk_rating,
    region,
    relationship_manager,
    valid_from,
    valid_to,
    is_current
FROM trainer_demo.demo_06.client_dim
WHERE client_id = 'CLT-00001'
ORDER BY valid_from;


In [0]:
%sql
-- Point-in-time query: what was a client's risk rating at the time of a specific trade?
-- Join the fact table to the dimension using the surrogate key's validity window.
-- This is the key query pattern enabled by SCD Type 2.

SELECT
    t.trade_id,
    t.trade_date,
    t.ticker,
    t.total_value,
    c.risk_rating    AS risk_at_trade_time,
    c.region         AS region_at_trade_time,
    c.relationship_manager
FROM trainer_demo.demo_06.trades AS t
JOIN trainer_demo.demo_06.accounts AS a
    ON t.account_id = a.account_id
JOIN trainer_demo.demo_06.client_dim AS c
    ON a.client_id = c.client_id
    AND CAST(t.trade_date AS TIMESTAMP) >= c.valid_from
    AND CAST(t.trade_date AS TIMESTAMP) <  c.valid_to
ORDER BY t.trade_date DESC
LIMIT 10;


## 시간에 따른 변화를 기록하기 위한 임시(기록) 테이블 설계 및 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-temporal-history-table.png)

### 🎯 데모: 거래 감사 추적을 위한 임시 기록 테이블
규제 기관은 거래 기록의 모든 변경 사항에 대한 완전한 감사 추적을 유지하기 위해 FinServ Demo를 요구합니다. 규정 준수 담당자가 "1월 15일 TRD-0000001 거래 상태는 어땠나요?"라고 묻는 경우 팀은 단순히 현재 가치를 반환하는 것이 아니라 정확하게 대답해야 합니다.
Delta Lake의 **변경 데이터 피드(CDF)**는 메타데이터를 사용하여 모든 행 수준 변경 사항(삽입, 업데이트, 삭제)을 캡처합니다. 구조적 스트리밍을 통해 채워진 영구 기록 테이블과 결합하면 변경할 수 없는 감사 추적이 생성됩니다.

In [0]:
%sql
-- Step 1: Enable change data feed on the trades table.
-- CDF records every insert, update, and delete with commit timestamps.
-- IMPORTANT: CDF only records changes from this point forward.

USE trainer_demo.demo_06;

ALTER TABLE trades
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Confirm CDF is enabled
SHOW TBLPROPERTIES trades;


In [0]:
%sql
-- Step 2: Make some changes to the trades table to generate CDF events.
-- Simulate a compliance correction: update failed trades and delete a cancelled one.

UPDATE trainer_demo.demo_06.trades
SET status = 'Settled'
WHERE status = 'Failed'
  AND trade_date < '2023-06-01';

DELETE FROM trainer_demo.demo_06.trades
WHERE trade_id = 'TRD-0000050' AND status = 'Pending';

SELECT COUNT(*) AS total_trades FROM trainer_demo.demo_06.trades;


In [0]:
%sql
-- Step 3: Query the change data feed to see what changed.
-- _change_type values:
--   'insert'           — new row was inserted
--   'update_preimage'  — values BEFORE the update
--   'update_postimage' — values AFTER the update
--   'delete'           — row was deleted

SELECT
    trade_id,
    status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('trainer_demo.demo_06.trades', 1)
ORDER BY _commit_timestamp, trade_id
LIMIT 30;


In [0]:
%sql
-- Step 4: Time travel — query the trades table as it was before the corrections.
-- This is useful for "as-of" regulatory reports and reconciliation.

-- View the full transaction history first
DESCRIBE HISTORY trainer_demo.demo_06.trades;


In [0]:
%sql
-- Query the trades table at version 0 (before any corrections).
-- Version 0 = the initial data load. Use this to compare current vs original.

SELECT status, COUNT(*) AS count
FROM trainer_demo.demo_06.trades VERSION AS OF 0
GROUP BY status
ORDER BY count DESC;


In [0]:
# Step 5: Persist the change history to a long-term audit table.
# Delta Lake's default retention is 7 days — not enough for financial compliance.
# Use Structured Streaming with readChangeFeed to build a permanent audit trail.

# Create destination audit table
spark.sql("""
    CREATE TABLE IF NOT EXISTS trainer_demo.demo_06.trades_audit_history
    USING DELTA
    AS SELECT
        trade_id, account_id, ticker, trade_type, total_value, trade_date, status,
        CAST(NULL AS STRING) AS _change_type,
        CAST(NULL AS BIGINT) AS _commit_version,
        CAST(NULL AS TIMESTAMP) AS _commit_timestamp
    FROM trainer_demo.demo_06.trades
    WHERE 1=0
""")

# Stream changes from trade table into audit history (trigger once = process all pending changes)
audit_query = (
    spark.readStream
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table("trainer_demo.demo_06.trades")
    .writeStream
        .option("checkpointLocation", "/tmp/checkpoints/trades_audit")
        .trigger(availableNow=True)
        .toTable("trainer_demo.demo_06.trades_audit_history")
)

audit_query.awaitTermination()
print(f"Audit rows captured: {spark.table('trainer_demo.demo_06.trades_audit_history').count():,}")


In [0]:
%sql
-- Step 6: Configure extended retention for compliance requirements.
-- Financial regulators often require 7+ years of audit data. This setting
-- ensures Delta time travel stays available longer than the default 7 days.

ALTER TABLE trainer_demo.demo_06.trades
SET TBLPROPERTIES (
    delta.logRetentionDuration         = 'interval 90 days',
    delta.deletedFileRetentionDuration = 'interval 90 days'
);

SHOW TBLPROPERTIES trainer_demo.demo_06.trades;


## 요구 사항에 따라 열이나 테이블의 세분성을 선택합니다.

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/9-understand-what-granularity-represents.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-granularity.png)

### 🎯 데모: 시장 데이터 및 거래 사실표의 세분성 선택
FinServ 데모는 사용 사례에 따라 다양한 세부 수준으로 시장 가격 데이터를 저장합니다.
| Use case | Needed granularity | Table |
|---|---|---|
| 일별 손익 보고서 | 티커별 일별 OHLCV | `market_data` (이미 일별 제공) |
| 성과 기여도 |  자산 클래스 별 월평균 종가| 사전 집계된 요약 테이블 |
| 틱 데이터 분석 | 초 단위 가격 변동 | 미포함 (스트리밍이 필요함) |

**키 인사이트**: 이 설정은 일별 세분화 수준으로 `market_data`를 생성합니다. 월별까지 집계는 가능하지만, 일중 데이터까지 세분화할 수는 없습니다. 예산이 허용하는 가장 세분화된 데이터를 확보하는 것이 좋습니다.

In [0]:
%sql
-- View daily granularity: each row = one ticker for one trading day
SELECT
    price_date,
    ticker,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    asset_class
FROM trainer_demo.demo_06.market_data
WHERE ticker = 'AAPL'
ORDER BY price_date
LIMIT 10;


In [0]:
%sql
-- Aggregate to monthly granularity for executive performance dashboards.
-- Once aggregated, intraday volatility information is lost — we only see
-- monthly open, high, low, close, and total volume.

SELECT
    DATE_TRUNC('MONTH', price_date)     AS month,
    ticker,
    FIRST_VALUE(open_price)
        OVER (PARTITION BY ticker, DATE_TRUNC('MONTH', price_date)
              ORDER BY price_date)       AS monthly_open,
    MAX(high_price)                      AS monthly_high,
    MIN(low_price)                       AS monthly_low,
    LAST_VALUE(close_price)
        OVER (PARTITION BY ticker, DATE_TRUNC('MONTH', price_date)
              ORDER BY price_date
              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS monthly_close,
    SUM(volume)                          AS monthly_volume
FROM trainer_demo.demo_06.market_data
WHERE ticker IN ('AAPL', 'MSFT', 'JPM')
GROUP BY month, ticker, price_date, open_price, close_price, high_price, low_price, volume
ORDER BY ticker, month
LIMIT 15;


In [0]:
%sql
-- Create a materialized view for the monthly aggregate.
-- Azure Databricks automatically refreshes this as the underlying market_data changes.
-- The grain: one row per ticker per month.

USE trainer_demo.demo_06;

CREATE OR REPLACE MATERIALIZED VIEW market_data_monthly
AS
SELECT
    DATE_TRUNC('MONTH', price_date)  AS price_month,
    ticker,
    asset_class,
    AVG(close_price)                 AS avg_close,
    MAX(high_price)                  AS period_high,
    MIN(low_price)                   AS period_low,
    SUM(volume)                      AS total_volume,
    COUNT(*)                         AS trading_days
FROM trainer_demo.demo_06.market_data
GROUP BY DATE_TRUNC('MONTH', price_date), ticker, asset_class;

SELECT * FROM market_data_monthly WHERE ticker = 'AAPL' ORDER BY price_month LIMIT 12;


In [0]:
%sql
-- Compare row counts at different granularity levels to illustrate storage impact.
-- Finer granularity = more rows = higher storage and compute cost.

SELECT 'Daily (atomic grain)' AS grain_level, COUNT(*) AS rows
FROM trainer_demo.demo_06.market_data

UNION ALL

SELECT 'Monthly aggregate' AS grain_level, COUNT(*) AS rows
FROM trainer_demo.demo_06.market_data_monthly

ORDER BY rows DESC;


## 관리되는 테이블과 관리되지 않는 테이블을 선택하세요

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-managed-vs-unmanaged-tables.png)

### 🎯 데모: 금융 서비스의 관리 테이블과 외부 테이블
FinServ 데모:
- **관리형 테이블**은 거래, 고객 데이터, 시장 가격 등 Databricks 레이크하우스에만 있는 모든 핵심 데이터에 사용됩니다. Unity 카탈로그는 자동 정리를 포함하여 데이터 수명주기를 제어합니다.
- **외부 테이블**은 위험 관리 팀의 온프레미스 시스템이 동일한 기본 Parquet 파일에 대한 읽기 액세스가 필요한 경우 또는 데이터를 이동하지 않고 레거시 Hive 관리 테이블을 마이그레이션할 때 사용됩니다.

> **표시할 주요 차이점**: 관리되는 테이블을 삭제하고 → 데이터가 제거되도록 예약됩니다. 외부 테이블을 삭제하고 → 데이터 파일은 그대로 유지됩니다.

---
#### 🔎 정렬 예시
- Databricks SQL에서 결과를 정렬하려면 `ORDER BY` 또는 `SORT BY`를 사용합니다.
- `ORDER BY`는 전체 결과를 완전히 정렬합니다.
- `SORT BY`는 각 파티션 내에서만 정렬합니다.

예시:
sql
SELECT * FROM portfolio_positions_managed
ORDER BY total_invested DESC;

sql
SELECT * FROM portfolio_positions_external
SORT BY account_id ASC;

In [0]:
%sql
-- Managed table: Unity Catalog owns both metadata AND data files.
-- When dropped, data files are automatically removed after 8 days (UNDROP is possible within that window).

USE trainer_demo.demo_06;

CREATE TABLE IF NOT EXISTS portfolio_positions_managed
USING DELTA
AS
SELECT
    a.account_id,
    t.ticker,
    t.asset_class,
    SUM(CASE WHEN t.trade_type = 'Buy' THEN t.quantity
             WHEN t.trade_type = 'Sell' THEN -t.quantity
             ELSE 0 END)   AS net_quantity,
    SUM(t.total_value)       AS total_invested
FROM trades t
JOIN accounts a ON t.account_id = a.account_id
GROUP BY a.account_id, t.ticker, t.asset_class;

-- DESCRIBE EXTENDED shows the storage location — it's in the Unity Catalog managed storage
DESCRIBE EXTENDED portfolio_positions_managed;


In [0]:
%sql
-- External table: Unity Catalog manages only metadata.
-- The data lives at an explicitly specified LOCATION (your ADLS path).
-- When dropped, the data files remain at that location.
--
-- Use case: the risk management system uses Apache Spark directly against the
-- same Parquet files — it doesn't go through Unity Catalog.

-- NOTE: Replace the LOCATION path with a valid external location registered in Unity Catalog.
-- This example uses the Volume path to simulate an external storage location.

CREATE TABLE IF NOT EXISTS portfolio_positions_external
USING DELTA
LOCATION '/Volumes/trainer_demo/demo_06/landing/portfolio_positions/'
AS
SELECT
    a.account_id,
    t.ticker,
    t.asset_class,
    SUM(CASE WHEN t.trade_type = 'Buy' THEN t.quantity
             WHEN t.trade_type = 'Sell' THEN -t.quantity
             ELSE 0 END)   AS net_quantity,
    SUM(t.total_value)       AS total_invested
FROM trades t
JOIN accounts a ON t.account_id = a.account_id
GROUP BY a.account_id, t.ticker, t.asset_class;

-- DESCRIBE EXTENDED shows the storage location points to YOUR specified path
DESCRIBE EXTENDED portfolio_positions_external;


In [0]:
%sql
-- Compare drop behavior to illustrate the key difference.
-- Show the tables exist first, then drop them.

SHOW TABLES IN trainer_demo.demo_06 LIKE 'portfolio_positions*';


In [0]:
%sql
-- Drop the managed table: metadata removed, data scheduled for deletion after 8 days.
-- You CAN undo this with UNDROP TABLE within the 8-day window.
DROP TABLE IF EXISTS trainer_demo.demo_06.portfolio_positions_managed;

-- Drop the external table: only metadata removed, data files REMAIN at the LOCATION.
-- There is no UNDROP for external tables.
DROP TABLE IF EXISTS trainer_demo.demo_06.portfolio_positions_external;

-- The files at /Volumes/trainer_demo/demo_06/landing/portfolio_positions/ still exist
-- even after the external table is dropped.
SELECT '외부 테이블이 삭제되었습니다. 데이터 파일은 LOCATION 경로에 여전히 존재합니다.' AS note;


## 클러스터링 전략 설계 및 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-clustering-strategy.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/11-understand-data-skipping-fundamentals.png)

### 🎯 데모: 다차원 거래 쿼리를 위한 리퀴드 클러스터링 및 정렬
FinServ Demo의 규정 준수 분석가는 다양한 방식으로 거래를 쿼리합니다.
- "2023년 1분기 모든 EMEA 거래" → `region` 및 `trade_date`을 필터링하고, 결과를 `trade_date`로 정렬합니다.
- "모든 JPM 주식 매입" → `ticker` AND `asset_class` AND `trade_type` 필터링, `total_value`로 내림차순 정렬.
- "모든 대규모 결제 거래" → `status` 및 `total_value`을 필터링, `total_value`로 정렬합니다.

기존 파티셔닝은 **1차원**에만 도움이 됩니다. **리퀴드 클러스터링**은 여러 필터 열에 동시에 데이터를 배치하여 세 가지 쿼리 패턴을 모두 빠르게 만듭니다.  
또한 Databricks SQL에서 결과를 정렬하려면 `ORDER BY` 또는 `SORT BY`를 사용합니다.

> **트레이너 팁**: `OPTIMIZE` 이후에는 다양한 필터 조합을 사용하는 쿼리가 모두 데이터 건너뛰기의 이점을 얻습니다. 즉, 파티션 키가 필요하지 않습니다.  
> 정렬이 필요한 경우 쿼리에서 `ORDER BY`를 활용하세요.

In [0]:
%sql
-- Create a liquid clustered version of the trades table.
-- CLUSTER BY up to 4 columns that appear most frequently in filters.
-- Unlike PARTITIONED BY, you can change clustering keys later without rewriting data.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE trades_clustered
USING DELTA
CLUSTER BY (trade_date, region, asset_class, status)
AS
SELECT * FROM trades;

DESCRIBE EXTENDED trades_clustered;


In [0]:
%sql
-- Run OPTIMIZE to physically apply liquid clustering to the data files.
-- Delta Lake reorganizes data so rows with similar filter-column values
-- are stored in the same files — enabling effective data skipping.

OPTIMIZE trainer_demo.demo_06.trades_clustered;


In [0]:
%sql
-- Query 1: EMEA trades in Q1 2023 — benefits from trade_date + region clustering
SELECT COUNT(*) AS emea_q1_trades, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_clustered
WHERE region = 'EMEA'
  AND trade_date BETWEEN '2023-01-01' AND '2023-03-31';


In [0]:
%sql
-- Query 2: Settled equity trades — benefits from asset_class + status clustering
SELECT ticker, trade_type, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_clustered
WHERE asset_class = 'Equity'
  AND status = 'Settled'
GROUP BY ticker, trade_type
ORDER BY total_value DESC
LIMIT 10;


In [0]:
%sql
-- Update clustering keys without rewriting the table.
-- Query patterns have changed: analysts now also filter frequently on ticker.
-- With liquid clustering, we can add ticker to the clustering strategy incrementally.

ALTER TABLE trainer_demo.demo_06.trades_clustered
CLUSTER BY (trade_date, region, ticker, status);

-- Next OPTIMIZE run will apply the new clustering going forward.
-- Previously written files are NOT automatically rewritten — use OPTIMIZE FULL for that.
-- OPTIMIZE trainer_demo.demo_06.trades_clustered FULL;   -- uncomment to rewrite all data

SELECT 'Clustering keys updated — next OPTIMIZE will reorganize new data' AS note;


In [0]:
%sql
-- Enabling deletion vectors improves UPDATE/DELETE performance on the trades table.
-- Instead of rewriting entire Parquet files for each correction, Delta Lake marks
-- modified rows in a separate deletion vector file and merges on read.
-- This is especially valuable for the frequent compliance corrections at FinServ Demo.

ALTER TABLE trainer_demo.demo_06.trades_clustered
SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Simulate a compliance correction — this is now much faster with deletion vectors
UPDATE trainer_demo.demo_06.trades_clustered
SET status = 'Settled'
WHERE status = 'Failed' AND trade_date < '2023-03-01';

DESCRIBE EXTENDED trainer_demo.demo_06.trades_clustered;
